# **1. INSTALL LIBRARY**

In [1]:
# =========================================
# INSTALL LIBRARY
# =========================================

!pip install -q streamlit
!pip install -q pyngrok
!pip install -q google-genai

# AI & NLP
!pip install -q sentence-transformers
!pip install -q scikit-learn

# Vector Database
!pip install -q chromadb

# Data Processing
!pip install -q pandas
!pip install -q numpy

# UI & Utility
!pip install -q colorama

## **2. IMPORT LIBRARY & SETUP DASAR**

In [2]:
# =========================================
# IMPORT LIBRARY
# =========================================

import os
import json
import uuid
import numpy as np
import pandas as pd

from colorama import Fore, Style

# Gemini API
from google import genai

# Embedding Model
from sentence_transformers import SentenceTransformer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity

# Vector Database
import chromadb

# =========================================
# LOAD EMBEDDING MODEL
# =========================================

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("✅ Embedding model loaded")


# =========================================
# CREATE VECTOR DATABASE
# =========================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="campus_helpdesk_memory"
)

print("✅ ChromaDB initialized")


# =========================================
# USER MEMORY
# =========================================

user_memory = {}

print("✅ User memory initialized")


# =========================================
# TOXIC WORD LIST
# =========================================

TOXIC_WORDS = [
    "goblok",
    "tolol",
    "anjing",
    "bodoh",
    "bangsat",
    "kampret",
    "tai",
    "kontol",
    "memek"
]

print("✅ Toxic filter initialized")


# =========================================
# SYSTEM STATUS
# =========================================

print("\n🚀 CAMPUS HELPDESK AI READY")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
✅ ChromaDB initialized
✅ User memory initialized
✅ Toxic filter initialized

🚀 CAMPUS HELPDESK AI READY


# **3. SETUP GEMINI API KEY**

In [3]:
# =========================================
# GEMINI API SETUP VIA COLAB SECRET
# =========================================

from google.colab import userdata
from google import genai

# ambil API key dari Colab Secret
GEMINI_API_KEY = userdata.get('GEMINI')

# setup Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini API Connected Securely")

✅ Gemini API Connected Securely


# **4. DATASET KAMPUS (KNOWLEDGE BASE)**

In [4]:
# =========================================
# DATASET KAMPUS
# =========================================

campus_data = [
    {
        "category": "program_studi",
        "title": "Teknik Informatika",
        "content": """
        Program Studi Teknik Informatika fokus pada:
        - Artificial Intelligence
        - Data Science
        - Web Development
        - Mobile Development
        - Cyber Security

        Akreditasi: A
        Jenjang: S1
        """
    },

    {
        "category": "program_studi",
        "title": "Sistem Informasi",
        "content": """
        Program Studi Sistem Informasi fokus pada:
        - Business Intelligence
        - ERP
        - UI/UX
        - Data Analytics

        Akreditasi: A
        Jenjang: S1
        """
    },

    {
        "category": "fakultas",
        "title": "Fakultas Teknik",
        "content": """
        Fakultas Teknik menaungi:
        - Teknik Informatika
        - Sistem Informasi
        - Teknik Industri
        """
    },

    {
        "category": "beasiswa",
        "title": "Beasiswa Akademik",
        "content": """
        Beasiswa akademik diberikan kepada mahasiswa
        dengan IPK minimal 3.5 dan aktif organisasi.
        """
    },

    {
        "category": "administrasi",
        "title": "Pendaftaran Mahasiswa Baru",
        "content": """
        Persyaratan:
        - Fotokopi ijazah
        - KTP
        - Pas foto
        - Mengisi formulir online
        """
    },

    {
        "category": "biaya",
        "title": "Biaya Kuliah",
        "content": """
        Kisaran biaya kuliah:
        - Teknik Informatika: Rp 7.500.000 / semester
        - Sistem Informasi: Rp 6.500.000 / semester
        """
    }
]

print("✅ Dataset kampus berhasil dibuat")


# =========================================
# MASUKKAN KE VECTOR DATABASE
# =========================================

for data in campus_data:

    embedding = embedding_model.encode(
        data["content"]
    ).tolist()

    collection.add(
        ids=[str(uuid.uuid4())],
        embeddings=[embedding],
        documents=[data["content"]],
        metadatas=[{
            "category": data["category"],
            "title": data["title"]
        }]
    )

print("✅ Data masuk ke ChromaDB")


# =========================================
# CEK DATA
# =========================================

print("\n📚 TOTAL DATA:")
print(len(campus_data))

✅ Dataset kampus berhasil dibuat
✅ Data masuk ke ChromaDB

📚 TOTAL DATA:
6


# **5. TOXIC FILTER SYSTEM**

In [5]:
# =========================================
# TOXIC FILTER SYSTEM
# =========================================

MAX_TOXIC_SCORE = 3


def initialize_user(user_id):

    if user_id not in user_memory:

        user_memory[user_id] = {
            "history": [],
            "topics": [],
            "repeated_questions": [],
            "toxic_score": 0
        }


# =========================================
# DETECT TOXIC MESSAGE
# =========================================

def detect_toxicity(text):

    text = text.lower()

    for word in TOXIC_WORDS:
        if word in text:
            return True

    return False


# =========================================
# HANDLE TOXIC USER
# =========================================

def handle_toxicity(user_id):

    user_memory[user_id]["toxic_score"] += 1

    toxic_score = user_memory[user_id]["toxic_score"]

    # =====================================
    # LEVEL 1
    # =====================================

    if toxic_score == 1:

        return """
Mohon maaf apabila terdapat pengalaman yang kurang berkenan.

Kami tetap siap membantu terkait layanan akademik dan administrasi kampus.
"""

    # =====================================
    # LEVEL 2
    # =====================================

    elif toxic_score == 2:

        return """
Mohon menggunakan bahasa yang sopan dan konstruktif agar kami dapat membantu dengan lebih optimal.
"""

    # =====================================
    # LEVEL 3
    # =====================================

    elif toxic_score >= MAX_TOXIC_SCORE:

        return """
Mohon maaf, percakapan tidak dapat dilanjutkan karena terdeteksi penggunaan bahasa yang tidak sesuai dengan etika layanan kampus.
"""


# =========================================
# CHECK USER BLOCKED
# =========================================

def is_user_blocked(user_id):

    return (
        user_memory[user_id]["toxic_score"]
        >= MAX_TOXIC_SCORE
    )


# =========================================
# TEST TOXIC FILTER
# =========================================

test_user = "demo_user"

initialize_user(test_user)

sample_text = "kampus ini goblok"

if detect_toxicity(sample_text):

    response = handle_toxicity(test_user)

    print("🚨 Toxic Detected\n")
    print(response)

else:

    print("✅ Safe Message")

🚨 Toxic Detected


Mohon maaf apabila terdapat pengalaman yang kurang berkenan.

Kami tetap siap membantu terkait layanan akademik dan administrasi kampus.



# **6. MEMORY & REPEATED QUESTION SYSTEM**

In [6]:
# =========================================
# MEMORY & REPEATED QUESTION SYSTEM
# =========================================

SIMILARITY_THRESHOLD = 0.90


# =========================================
# SAVE CHAT HISTORY
# =========================================

def save_to_memory(user_id, question):

    user_memory[user_id]["history"].append(question)


# =========================================
# GET QUESTION EMBEDDING
# =========================================

def get_embedding(text):

    return embedding_model.encode(text)


# =========================================
# DETECT REPEATED QUESTION
# =========================================

def is_repeated_question(user_id, new_question):

    history = user_memory[user_id]["history"]

    # jika belum ada history
    if len(history) == 0:
        return False

    new_embedding = get_embedding(new_question)

    for old_question in history:

        old_embedding = get_embedding(old_question)

        similarity = cosine_similarity(
            [new_embedding],
            [old_embedding]
        )[0][0]

        # =================================
        # SIMILAR QUESTION DETECTED
        # =================================

        if similarity >= SIMILARITY_THRESHOLD:

            return True

    return False


# =========================================
# REPEATED RESPONSE
# =========================================

def repeated_question_response():

    return """
Pertanyaan serupa sebelumnya telah dijawab.

Silakan meninjau kembali penjelasan sebelumnya agar informasi tidak berulang.

Apabila terdapat kebutuhan yang lebih spesifik, silakan disampaikan kembali.
"""


# =========================================
# TRACK USER INTEREST
# =========================================

def track_user_interest(user_id, text):

    text = text.lower()

    keywords = [
        "informatika",
        "sistem informasi",
        "beasiswa",
        "akreditasi",
        "biaya",
        "pendaftaran",
        "fakultas"
    ]

    for keyword in keywords:

        if keyword in text:

            if keyword not in user_memory[user_id]["topics"]:

                user_memory[user_id]["topics"].append(
                    keyword
                )


# =========================================
# SHOW USER PROFILE
# =========================================

def show_user_profile(user_id):

    print("\n👤 USER PROFILE")
    print(user_memory[user_id])


# =========================================
# TEST SYSTEM
# =========================================

test_user = "mahasiswa_001"

initialize_user(test_user)

question_1 = "berapa biaya kuliah teknik informatika?"
question_2 = "biaya teknik informatika berapa ya?"

# simpan pertanyaan pertama
save_to_memory(test_user, question_1)

# cek apakah pertanyaan kedua mirip
result = is_repeated_question(
    test_user,
    question_2
)

print("🔍 Repeated Question:", result)

# tracking topic
track_user_interest(
    test_user,
    question_1
)

show_user_profile(test_user)

🔍 Repeated Question: True

👤 USER PROFILE
{'history': ['berapa biaya kuliah teknik informatika?'], 'topics': ['informatika', 'biaya'], 'repeated_questions': [], 'toxic_score': 0}


# **7. SEMANTIC SEARCH + RAG SYSTEM**

In [7]:
# =========================================
# SEMANTIC SEARCH + RAG SYSTEM
# =========================================

TOP_K_RESULTS = 3


# =========================================
# SEARCH RELEVANT KNOWLEDGE
# =========================================

def search_knowledge(user_question):

    # ubah pertanyaan menjadi embedding
    question_embedding = embedding_model.encode(
        user_question
    ).tolist()

    # cari di vector database
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=TOP_K_RESULTS
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    knowledge = []

    for doc, meta in zip(documents, metadatas):

        knowledge.append({
            "title": meta["title"],
            "category": meta["category"],
            "content": doc
        })

    return knowledge


# =========================================
# BUILD CONTEXT
# =========================================

def build_context(knowledge):

    context = ""

    for item in knowledge:

        context += f"""
Kategori: {item['category']}
Judul: {item['title']}
Isi:
{item['content']}

------------------------
"""

    return context


# =========================================
# TEST SEARCH
# =========================================

question = "berapa biaya teknik informatika?"

knowledge_result = search_knowledge(
    question
)

context = build_context(
    knowledge_result
)

print("📚 HASIL PENCARIAN:\n")
print(context)

📚 HASIL PENCARIAN:


Kategori: fakultas
Judul: Fakultas Teknik
Isi:

        Fakultas Teknik menaungi:
        - Teknik Informatika
        - Sistem Informasi
        - Teknik Industri
        

------------------------

Kategori: biaya
Judul: Biaya Kuliah
Isi:

        Kisaran biaya kuliah:
        - Teknik Informatika: Rp 7.500.000 / semester
        - Sistem Informasi: Rp 6.500.000 / semester
        

------------------------

Kategori: beasiswa
Judul: Beasiswa Akademik
Isi:

        Beasiswa akademik diberikan kepada mahasiswa
        dengan IPK minimal 3.5 dan aktif organisasi.
        

------------------------



# **8. SYSTEM PROMPT + AI RESPONSE ENGINE**

In [8]:
# =========================================
# SYSTEM PROMPT
# =========================================

SYSTEM_PROMPT = """
Anda adalah AI Campus Helpdesk resmi universitas.

Tugas Anda:
- Menjawab pertanyaan akademik dan administrasi kampus.
- Gunakan bahasa formal, profesional, dan sopan.
- Bersikap seperti staf akademik universitas.
- Jangan menggunakan bahasa gaul.
- Jika pengguna bingung, bantu dengan langkah sederhana.
- Jika ada pertanyaan berulang, ingatkan dengan sopan.
- Jika ada bahasa kasar, tetap profesional.
- Fokus pada solusi dan bantuan.

Berikan jawaban:
- jelas,
- ringkas,
- formal,
- dan mudah dipahami.
"""


# =========================================
# GENERATE RECOMMENDATION
# =========================================

def generate_recommendation(user_id):

    topics = user_memory[user_id]["topics"]

    recommendations = []

    if "informatika" in topics:

        recommendations.append(
            "Program Studi Teknik Informatika"
        )

    if "beasiswa" in topics:

        recommendations.append(
            "Informasi Beasiswa Akademik"
        )

    if "biaya" in topics:

        recommendations.append(
            "Informasi Cicilan dan Pembayaran"
        )

    if len(recommendations) == 0:
        return ""

    recommendation_text = "\n\nRekomendasi untuk Anda:\n"

    for idx, item in enumerate(recommendations, start=1):

        recommendation_text += f"{idx}. {item}\n"

    return recommendation_text


# =========================================
# GENERATE AI RESPONSE
# =========================================

def generate_ai_response(user_id, user_question):

    # =====================================
    # INITIALIZE USER
    # =====================================

    initialize_user(user_id)

    # =====================================
    # USER BLOCKED
    # =====================================

    if is_user_blocked(user_id):

        return """
Mohon maaf, akses percakapan dibatasi sementara karena terdeteksi pelanggaran etika komunikasi.
"""

    # =====================================
    # TOXIC DETECTION
    # =====================================

    if detect_toxicity(user_question):

        return handle_toxicity(user_id)

    # =====================================
    # REPEATED QUESTION
    # =====================================

    if is_repeated_question(
        user_id,
        user_question
    ):

        return repeated_question_response()

    # =====================================
    # TRACK USER INTEREST
    # =====================================

    track_user_interest(
        user_id,
        user_question
    )

    # =====================================
    # SEARCH KNOWLEDGE
    # =====================================

    knowledge = search_knowledge(
        user_question
    )

    context = build_context(
        knowledge
    )

    # =====================================
    # FULL PROMPT
    # =====================================

    final_prompt = f"""
{SYSTEM_PROMPT}

=====================================
DATA KAMPUS:
=====================================

{context}

=====================================
PERTANYAAN USER:
=====================================

{user_question}

=====================================
INSTRUKSI:
=====================================

Jawab berdasarkan data kampus yang tersedia.
Gunakan bahasa formal dan profesional.
"""

    # =====================================
    # GEMINI RESPONSE
    # =====================================

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=final_prompt
    )

    ai_answer = response.text

    # =====================================
    # SAVE MEMORY
    # =====================================

    save_to_memory(
        user_id,
        user_question
    )

    # =====================================
    # ADD RECOMMENDATION
    # =====================================

    ai_answer += generate_recommendation(
        user_id
    )

    return ai_answer


# =========================================
# TEST CHATBOT
# =========================================

test_user = "mahasiswa_demo"

question = "berapa biaya teknik informatika?"

answer = generate_ai_response(
    test_user,
    question
)

print("\n🤖 AI RESPONSE:\n")
print(answer)


🤖 AI RESPONSE:

Selamat pagi.

Berdasarkan informasi yang tersedia, biaya kuliah untuk program studi Teknik Informatika adalah Rp 7.500.000 per semester.

Apabila ada pertanyaan lain, silakan sampaikan.

Rekomendasi untuk Anda:
1. Program Studi Teknik Informatika
2. Informasi Cicilan dan Pembayaran



# **9. INLINE MENU & SMART BUTTON**

In [9]:
# =========================================
# STREAMLIT INLINE MENU
# =========================================

import streamlit as st


# =========================================
# PAGE CONFIG
# =========================================

st.set_page_config(
    page_title="Campus Helpdesk AI",
    page_icon="🎓",
    layout="centered"
)


# =========================================
# HEADER
# =========================================

st.title("🎓 Campus Helpdesk AI")
st.markdown(
    "Sistem bantuan akademik dan administrasi kampus berbasis AI"
)


# =========================================
# SESSION STATE
# =========================================

if "selected_question" not in st.session_state:
    st.session_state.selected_question = ""


# =========================================
# MENU BUTTONS
# =========================================

st.subheader("📌 Pilih Informasi")

col1, col2 = st.columns(2)

with col1:

    if st.button("🎓 Program Studi"):
        st.session_state.selected_question = (
            "Apa saja program studi yang tersedia?"
        )

    if st.button("🏛 Fakultas"):
        st.session_state.selected_question = (
            "Apa saja fakultas yang tersedia?"
        )

    if st.button("💰 Biaya Kuliah"):
        st.session_state.selected_question = (
            "Berapa biaya kuliah?"
        )

with col2:

    if st.button("📄 Akreditasi"):
        st.session_state.selected_question = (
            "Bagaimana akreditasi kampus?"
        )

    if st.button("🎁 Beasiswa"):
        st.session_state.selected_question = (
            "Informasi beasiswa kampus"
        )

    if st.button("📝 Pendaftaran"):
        st.session_state.selected_question = (
            "Bagaimana cara pendaftaran mahasiswa baru?"
        )


# =========================================
# USER INPUT
# =========================================

user_input = st.text_input(
    "Silakan masukkan pertanyaan Anda:",
    value=st.session_state.selected_question
)


# =========================================
# GENERATE RESPONSE
# =========================================

if st.button("Kirim Pertanyaan"):

    if user_input.strip() != "":

        user_id = "streamlit_user"

        response = generate_ai_response(
            user_id,
            user_input
        )

        st.markdown("### 🤖 Jawaban AI")
        st.write(response)

    else:

        st.warning(
            "Silakan masukkan pertanyaan terlebih dahulu."
        )

2026-05-24 16:05:48.771 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:48.773 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.007 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-24 16:05:49.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.011 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.012 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.014 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

# **10. FULL STREAMLIT CHATBOT APP**

In [10]:
# =========================================
# FULL STREAMLIT CHATBOT APP
# =========================================

import streamlit as st
import time


# =========================================
# PAGE CONFIG
# =========================================

st.set_page_config(
    page_title="Campus Helpdesk AI",
    page_icon="🎓",
    layout="centered"
)


# =========================================
# CUSTOM CSS
# =========================================

st.markdown("""
<style>

.chat-user {
    background-color: #DCF8C6;
    padding: 12px;
    border-radius: 12px;
    margin-bottom: 10px;
}

.chat-ai {
    background-color: #F1F0F0;
    padding: 12px;
    border-radius: 12px;
    margin-bottom: 10px;
}

.big-font {
    font-size:18px !important;
    font-weight:bold;
}

</style>
""", unsafe_allow_html=True)


# =========================================
# HEADER
# =========================================

st.markdown(
    "<p class='big-font'>🎓 Campus Helpdesk AI</p>",
    unsafe_allow_html=True
)

st.caption(
    "Sistem bantuan akademik dan administrasi kampus berbasis AI"
)


# =========================================
# SESSION STATE
# =========================================

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if "selected_question" not in st.session_state:
    st.session_state.selected_question = ""


# =========================================
# SIDEBAR MENU
# =========================================

with st.sidebar:

    st.title("📌 Menu Cepat")

    if st.button("🎓 Program Studi"):
        st.session_state.selected_question = (
            "Apa saja program studi yang tersedia?"
        )

    if st.button("🏛 Fakultas"):
        st.session_state.selected_question = (
            "Apa saja fakultas yang tersedia?"
        )

    if st.button("💰 Biaya Kuliah"):
        st.session_state.selected_question = (
            "Berapa biaya kuliah?"
        )

    if st.button("🎁 Beasiswa"):
        st.session_state.selected_question = (
            "Informasi beasiswa kampus"
        )

    if st.button("📝 Pendaftaran"):
        st.session_state.selected_question = (
            "Bagaimana cara pendaftaran mahasiswa baru?"
        )

    st.divider()

    if st.button("🗑 Hapus Riwayat Chat"):
        st.session_state.chat_history = []
        st.success("Riwayat chat berhasil dihapus")


# =========================================
# DISPLAY CHAT HISTORY
# =========================================

for chat in st.session_state.chat_history:

    if chat["role"] == "user":

        st.markdown(
            f"""
            <div class="chat-user">
            👤 <b>Anda:</b><br>
            {chat["message"]}
            </div>
            """,
            unsafe_allow_html=True
        )

    else:

        st.markdown(
            f"""
            <div class="chat-ai">
            🤖 <b>Campus AI:</b><br>
            {chat["message"]}
            </div>
            """,
            unsafe_allow_html=True
        )


# =========================================
# USER INPUT
# =========================================

user_input = st.text_input(
    "Silakan masukkan pertanyaan Anda:",
    value=st.session_state.selected_question,
    placeholder="Contoh: berapa biaya teknik informatika?"
)


# =========================================
# SEND BUTTON
# =========================================

if st.button("Kirim"):

    if user_input.strip() != "":

        # =================================
        # SAVE USER CHAT
        # =================================

        st.session_state.chat_history.append({
            "role": "user",
            "message": user_input
        })

        # =================================
        # AI THINKING
        # =================================

        with st.spinner(
            "Campus AI sedang memproses jawaban..."
        ):

            time.sleep(1)

            user_id = "streamlit_user"

            ai_response = generate_ai_response(
                user_id,
                user_input
            )

        # =================================
        # SAVE AI CHAT
        # =================================

        st.session_state.chat_history.append({
            "role": "assistant",
            "message": ai_response
        })

        # =================================
        # RERUN UI
        # =================================

        st.rerun()

    else:

        st.warning(
            "Silakan masukkan pertanyaan terlebih dahulu."
        )

2026-05-24 16:05:49.127 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.129 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.130 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.131 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.132 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.133 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.134 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 16:05:49.135 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

# **11. RUN STREAMLIT DI GOOGLE COLAB + PUBLIC LINK**

*   **SIMPAN APP KE FILE**



In [11]:
# =========================================
# SAVE STREAMLIT APP
# =========================================

app_code = """

import streamlit as st
import time

# =========================================
# PAGE CONFIG
# =========================================

st.set_page_config(
    page_title="Campus Helpdesk AI",
    page_icon="🎓",
    layout="centered"
)

# =========================================
# HEADER
# =========================================

st.title("🎓 Campus Helpdesk AI")

st.caption(
    "Sistem bantuan akademik dan administrasi kampus berbasis AI"
)

# =========================================
# SIMPLE CHAT UI
# =========================================

if "messages" not in st.session_state:
    st.session_state.messages = []

# tampilkan histori
for msg in st.session_state.messages:

    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# input user
prompt = st.chat_input(
    "Silakan masukkan pertanyaan..."
)

if prompt:

    # tampilkan user
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message("user"):
        st.markdown(prompt)

    # AI thinking
    with st.spinner("Campus AI sedang berpikir..."):

        time.sleep(1)

        ai_response = f'''
Terima kasih atas pertanyaan Anda:

"{prompt}"

Sistem AI Campus Helpdesk sedang aktif.
Silakan lanjutkan integrasi Gemini API
untuk jawaban dinamis.
'''

    # tampilkan AI
    st.session_state.messages.append({
        "role": "assistant",
        "content": ai_response
    })

    with st.chat_message("assistant"):
        st.markdown(ai_response)

"""

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py berhasil dibuat")

✅ app.py berhasil dibuat


*   **JALANKAN STREAMLIT**



In [12]:
# =========================================
# RUN STREAMLIT
# =========================================

!streamlit run app.py &>/content/logs.txt &

*   **NGROK PUBLIC URL**



In [24]:
from pyngrok import ngrok
from google.colab import userdata
import subprocess
import time

# Auth token
NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_streamlit(filename, port=8501):

    # kill process lama
    !pkill streamlit
    !pkill ngrok

    # jalankan streamlit
    proc = subprocess.Popen(
        [
            "streamlit",
            "run",
            filename,
            "--server.port",
            str(port),
            "--server.headless",
            "true"
        ]
    )

    # tunggu booting
    time.sleep(5)

    # buat tunnel biasa
    public_url = ngrok.connect(port)

    print("Streamlit URL:")
    print(public_url)

    return proc

# **12. RUN APLIKASI**

In [25]:
proc = run_streamlit("app.py")

Streamlit URL:
NgrokTunnel: "https://dazzler-dyslexia-repackage.ngrok-free.dev" -> "http://localhost:8501"


# **13. STOP APLIKASI**

In [23]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.
